# 01 Describing data: averages, spreads, medians

simplydata's words for describing one column of numbers, checked on the nine univariate datasets of the NIST
Statistical Reference Datasets (StRD). NIST certifies the mean, the standard deviation and the lag-1
autocorrelation of each to 15 significant digits.

**Only simplydata runs in this notebook.** The reference values are in `references/01_describing_data.csv`, written
by `scripts/references/01_describing_data.py`:
- the NIST certified values, compared by the log relative error (LRE: how many significant digits agree); a word
  must get at least 9 digits;
- NIST certifies no median, so the median is compared with R `median()` (run once by the reference script, relative
  difference 1e-12), and for NumAcc4 with its value exact by construction.

NumAcc1 to NumAcc4 are made hard on purpose: their values differ only in the last few digits of large numbers, which
defeats a careless one-pass formula for the standard deviation. Each check prints its source with a link.

In [1]:
import simplydata as sdata
from simplydata import SimplydataError

from checks import ReferenceChecks, data_file, quiet, read_rows_by, versions

NOTEBOOK = "01_describing_data"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)
NIST_UNIVARIATE = ["PiDigits", "Lottery", "Lew", "Mavro", "Michelso", "NumAcc1", "NumAcc2", "NumAcc3", "NumAcc4"]


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


data = {name: load(f"nist_{name}") for name in NIST_UNIVARIATE}

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The data

Each file holds one column of numbers (y). Difficulty as NIST rates it: PiDigits, Lottery, Lew, Mavro and Michelso
lower; NumAcc1 lower; NumAcc2 and NumAcc3 average; NumAcc4 higher.

In [2]:
for name, table in data.items():
    print(f"{name}: {quiet(table.count_rows)} values, the first {quiet(table.get_cell_at, row=1, column='y')}")

PiDigits: 5000 values, the first 3.0
Lottery: 218 values, the first 162.0
Lew: 200 values, the first -213.0
Mavro: 50 values, the first 2.0018
Michelso: 100 values, the first 299.85
NumAcc1: 3 values, the first 10000001.0
NumAcc2: 1001 values, the first 1.2
NumAcc3: 1001 values, the first 1000000.2
NumAcc4: 1001 values, the first 10000000.2


## Average: `table.find_average_of`

The same underneath: pandas `Series.mean()`.

In [3]:
# what the word prints, once (inside the loops below it runs quietly)
data["Michelso"].find_average_of("y")

The average of "y" is 299.85 (100 values).


299.8524

In [4]:
for name, table in data.items():
    checks.check(f"NIST {name}", "find_average_of", "mean", quiet(table.find_average_of, "y"))

agrees  NIST PiDigits | find_average_of | mean: simplydata 4.5348, NIST certified 4.5348; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/PiDigits.dat
agrees  NIST Lottery | find_average_of | mean: simplydata 518.958715596, NIST certified 518.958715596; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Lottery.dat
agrees  NIST Lew | find_average_of | mean: simplydata -177.435, NIST certified -177.435; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Lew.dat
agrees  NIST Mavro | find_average_of | mean: simplydata 2.001856, NIST certified 2.001856; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Mavro.dat
agrees  NIST Michelso | find_average_of | mean: simplydata 299.8524, NIST certified 299.8524; LRE 15.0 (needs >= 9)
      Source: NIST c

## Standard deviation: `table.find_standard_deviation_of`

The sample standard deviation (divide by n - 1). The same underneath: pandas `Series.std()`.

In [5]:
data["Michelso"].find_standard_deviation_of("y")

The standard deviation of "y" is 0.08 (100 values).


0.07901054781905067

In [6]:
for name, table in data.items():
    checks.check(f"NIST {name}", "find_standard_deviation_of", "standard deviation",
                 quiet(table.find_standard_deviation_of, "y"))

agrees  NIST PiDigits | find_standard_deviation_of | standard deviation: simplydata 2.86733906029, NIST certified 2.86733906029; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/PiDigits.dat
agrees  NIST Lottery | find_standard_deviation_of | standard deviation: simplydata 291.699727471, NIST certified 291.699727471; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Lottery.dat
agrees  NIST Lew | find_standard_deviation_of | standard deviation: simplydata 277.332168044, NIST certified 277.332168044; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Lew.dat
agrees  NIST Mavro | find_standard_deviation_of | standard deviation: simplydata 0.000429123454003, NIST certified 0.000429123454003; LRE 12.7 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Mavr

## Lag-1 autocorrelation: `table.find_autocorrelation_of`

How much each value is like the one before it. NIST certifies r(1) as
sum((y[i] - mean)(y[i+1] - mean)) / sum((y[i] - mean)^2), the definition of R's `acf()` and statsmodels' `acf()`.

NumAcc1 has only 3 values; `find_autocorrelation_of` asks for at least 4, so that one is recorded as SKIP with the
reason (from the reference file).

In [7]:
data["Michelso"].find_autocorrelation_of("y", rows_back=1)

"y" is most like itself 1 row back (correlation 0.535). 1 of the 1 correlations are outside the band expected from random noise (+/- 0.196). Ljung-Box test up to 1 rows back: fewer than 1 in 1000 studies would show this much if the readings were truly not linked to the readings before them (p < 0.0001).
Method: correlation of the column with itself 1 to 1 rows back, as R acf(x, lag.max = 1) (statsmodels); band +/- 1.96 / sqrt(n), as the dashed lines of R plot.acf; Ljung-Box as R Box.test(x, lag = 1, type = "Ljung-Box").


,rows back,correlation,band low,band high,outside band,Ljung-Box statistic,Ljung-Box p-value
1,1,0.5352,-0.195996,0.195996,yes,29.511865,5.557545e-08


In [8]:
for name, table in data.items():
    try:
        found = quiet(table.find_autocorrelation_of, "y", rows_back=1)
    except SimplydataError as refused:
        print(f"{name}: simplydata gives no number: {str(refused).splitlines()[0]}")
        continue
    checks.check(f"NIST {name}", "find_autocorrelation_of", "lag-1 autocorrelation",
                 read_rows_by(found, "rows back")[1]["correlation"])

agrees  NIST PiDigits | find_autocorrelation_of | lag-1 autocorrelation: simplydata -0.00355099287238, NIST certified -0.00355099287238; LRE 13.8 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/PiDigits.dat


agrees  NIST Lottery | find_autocorrelation_of | lag-1 autocorrelation: simplydata -0.120948622967, NIST certified -0.120948622967; LRE 14.9 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Lottery.dat
agrees  NIST Lew | find_autocorrelation_of | lag-1 autocorrelation: simplydata -0.307304800606, NIST certified -0.307304800606; LRE 14.8 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Lew.dat
agrees  NIST Mavro | find_autocorrelation_of | lag-1 autocorrelation: simplydata 0.937989183438, NIST certified 0.937989183438; LRE 14.1 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Mavro.dat
agrees  NIST Michelso | find_autocorrelation_of | lag-1 autocorrelation: simplydata 0.535199668621, NIST certified 0.535199668621; LRE 13.4 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/univ/data/Mic

## Median: `table.find_median_of`

NIST certifies no median, so the median is compared with R `median()`. For NumAcc4 it is known exactly by
construction: 500 values 0.1 below 10000000.2, 500 values 0.1 above, and one equal to it.

In [9]:
data["Michelso"].find_median_of("y")

The median of "y" is 299.85 (100 values).


299.85

In [10]:
for name, table in data.items():
    checks.check(f"NIST {name}", "find_median_of", "median", quiet(table.find_median_of, "y"))

agrees  NIST PiDigits | find_median_of | median: simplydata 5, R median 5; relative difference 0.00e+00 (allowed 1e-12)
      Source: R median (run reference) - https://search.r-project.org/R/refmans/stats/html/median.html
agrees  NIST Lottery | find_median_of | median: simplydata 522.5, R median 522.5; relative difference 0.00e+00 (allowed 1e-12)
      Source: R median (run reference) - https://search.r-project.org/R/refmans/stats/html/median.html
agrees  NIST Lew | find_median_of | median: simplydata -162, R median -162; relative difference 0.00e+00 (allowed 1e-12)
      Source: R median (run reference) - https://search.r-project.org/R/refmans/stats/html/median.html
agrees  NIST Mavro | find_median_of | median: simplydata 2.0018, R median 2.0018; relative difference 0.00e+00 (allowed 1e-12)
      Source: R median (run reference) - https://search.r-project.org/R/refmans/stats/html/median.html
agrees  NIST Michelso | find_median_of | median: simplydata 299.85, R median 299.85; relative

## Result

In [11]:
checks.finish()

skipped  NIST NumAcc1 | find_autocorrelation_of | lag-1 autocorrelation against NIST certified: simplydata gives no number: find_autocorrelation_of asks for at least 4 values and NumAcc1 has 3
01_describing_data: 37 comparisons - 35 agrees, 1 explained difference – precision limit, 1 skipped
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,01_describing_data,NIST PiDigits,find_average_of,mean,4.534800e+00,4.534800e+00,NIST certified,0.000000e+00,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,01_describing_data,NIST Lottery,find_average_of,mean,5.189587e+02,5.189587e+02,NIST certified,3.410605e-13,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,01_describing_data,NIST Lew,find_average_of,mean,-1.774350e+02,-1.774350e+02,NIST certified,0.000000e+00,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,01_describing_data,NIST Mavro,find_average_of,mean,2.001856e+00,2.001856e+00,NIST certified,-4.440892e-16,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,01_describing_data,NIST Michelso,find_average_of,mean,2.998524e+02,2.998524e+02,NIST certified,0.000000e+00,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
5,01_describing_data,NIST NumAcc1,find_average_of,mean,1.000000e+07,1.000000e+07,NIST certified,0.000000e+00,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
6,01_describing_data,NIST NumAcc2,find_average_of,mean,1.200000e+00,1.200000e+00,NIST certified,0.000000e+00,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
7,01_describing_data,NIST NumAcc3,find_average_of,mean,1.000000e+06,1.000000e+06,NIST certified,1.164153e-10,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
8,01_describing_data,NIST NumAcc4,find_average_of,mean,1.000000e+07,1.000000e+07,NIST certified,1.862645e-09,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
9,01_describing_data,NIST PiDigits,find_standard_deviation_of,standard deviation,2.867339e+00,2.867339e+00,NIST certified,-1.332268e-15,9.000000e+00,lre,15.000000,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/univ/data...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
